# Text analytics: MediaPipe.NET, GraviText and HF.Net
**EN** — Score reviews as they stream: MediaPipe.NET sentiment and language detection, GraviText lexicon sentiment, and (optionally) any Hugging Face classifier through HF.Net.

**ID** — Menilai ulasan secara langsung: sentimen dan deteksi bahasa dengan MediaPipe.NET, sentimen leksikon GraviText, dan (opsional) classifier Hugging Face apa pun lewat HF.Net.

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

In [ ]:
// Using a local build instead of nuget.org? Uncomment and point to <repo>/artifacts/nuget (absolute path):
// #i "nuget: C:/src/bigpipe/artifacts/nuget"
#r "nuget: BigPipe.Analytics.Gravicode, 0.1.0"
using BigPipe.Client;
using BigPipe.Client.Admin;
using BigPipe.Analytics;
var bootstrap = Environment.GetEnvironmentVariable("BIGPIPE_BOOTSTRAP") ?? "localhost:9092";
var admin = new BigPipeAdminClient(Environment.GetEnvironmentVariable("BIGPIPE_ADMIN") ?? "http://localhost:9644");
var http = new BigPipeHttpClient(Environment.GetEnvironmentVariable("BIGPIPE_HTTP") ?? "http://localhost:8082");
string Unique(string p) => $"nb-{p}-{Guid.NewGuid().ToString("N")[..6]}";
var cluster = await admin.GetClusterAsync();
Console.WriteLine($"Connected to {cluster.ClusterId} (BigPipe {cluster.Version}) — {cluster.Credit}");

In [ ]:
using BigPipe.Analytics.Gravicode;
var topic = Unique("reviews");
await admin.CreateTopicAsync(topic, 1);
string[] reviews =
[
    "Absolutely love this batik shirt, the quality is excellent!",
    "Terrible experience, the package arrived broken.",
    "Kopinya enak sekali dan pengirimannya cepat, pasti beli lagi!",
    "Great value for the price, my kids are happy with it.",
    "Worst purchase ever, a complete waste of money.",
];
await http.ProduceAsync(topic, reviews.Select((t, i) => new HttpProduceRecord { Key = $"r{i}", Value = new { text = t } }));
var sentiment = new MediaPipeSentiment();     // IDisposable: dispose when done
var language = new MediaPipeLanguageDetector();
var lexicon = new GraviSentiment();
var scored = new List<object>();
await foreach (var r in BigPipeSource.Kafka(bootstrap, topic, from: OffsetReset.Earliest))
{
    var text = r.Str("text")!;
    var (label, score) = sentiment.Analyze(text);
    scored.Add(new { lang = language.Detect(text).Language, mediapipe = label, score = Math.Round(score, 2), gravitext = lexicon.Analyze(text).Label, text });
    if (scored.Count == reviews.Length) break;
}
scored

### Optional: a Hugging Face model via HF.Net / Opsional: model Hugging Face lewat HF.Net
Downloads the model once to the local cache. Set `RUN_HF=1` to run this cell.

In [ ]:
if (Environment.GetEnvironmentVariable("RUN_HF") == "1")
{
    using var hf = HfTextModel.Load("distilbert-base-uncased-finetuned-sst-2-english");
    display(reviews.Select(t => new { t, prediction = hf.Classify(t) }).ToList());
}
else
{
    display("Skipped (set RUN_HF=1 to download and run a Hugging Face classifier).");
}